In [ ]:
import os
from typing import Any, cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.stats import zscore

from tabletop_py.gaze.preprocess import (
    clean_marker_data,
    format_marker_columns,
    verify_timestamps,
)

In [ ]:
SESSION_NAME = "session_2026-01-28_23-44-36"
MARKER_IDX = 0
SYNC_TRIGGER = "off"
DIM = "x"
USE_GRADIENT = False

In [ ]:
config = os.path.join(
    os.environ["TABLETOP_DIR"], "config", "gaze_estimation.yaml"
)
with open(config, "r") as f:
    config = cast(dict[str, Any], yaml.safe_load(f))

markers_freq = config["markers_freq"]
config = cast(dict[str, Any], config["preprocess"])

session_dir = os.path.join(os.environ["ROS_BAG_DIR"], SESSION_NAME)

markers_path = os.path.join(session_dir, "markers.csv")
teensy_path = os.path.join(session_dir, "teensy_sensor.csv")

raw_markers_df = pd.read_csv(markers_path, index_col=False)
raw_teensy_df = pd.read_csv(teensy_path, index_col=False)

In [ ]:
markers_df = format_marker_columns(
    raw_markers_df,
    marker_idx=MARKER_IDX,
    freq=markers_freq,
    **config["format_marker"],
)

markers_df = clean_marker_data(markers_df, **config["clean_marker"])
markers_df

In [ ]:
# TEENSY_DATA_COLS = [
#     "is_safety_laser_broken",
#     "is_left_arm_locked",
#     "is_right_arm_locked",
#     "is_reward_active",
#     "is_button_pressed",
#     "is_smartglass_revealed",
#     "left_tactile_glove_states",
#     "right_tactile_glove_states",
#     "sync_pulse_state",
#     "sync_pulse_last_time_on",
#     "sync_pulse_last_time_off",
#     "safety_laser_last_time_broken",
#     "button_last_time_pressed",
# ]
TEENSY_DATA_COLS = [
    "sync_pulse_last_time_on",
    "sync_pulse_last_time_off",
]


def format_teensy_columns(
    df: pd.DataFrame,
    *,
    freq: float,
    verify: bool = True,
    freq_rtol: float | None = None,
    freq_var_tol: float | None = None,
) -> pd.DataFrame:
    """
    Formats the timestamps to seconds and checks for monotonicity.

    Args:
        df: The teensy sensor data.
        freq: The expected frequency of the data.
        verify: Whether to verify the timestamps and monotonicity.
        freq_rtol: The relative tolerance for the frequency.
        freq_var_tol: The absolute tolerance for the frequency.

    Returns:
        The formatted eye tracker data.
    """
    df = df.copy()

    # Convert ROS timestamp to seconds
    df["time"] = df["header.stamp.sec"] + df["header.stamp.nanosec"] / 1e9
    df["sync_pulse_last_time_on"] = (
        df["sync_pulse_last_time_on.sec"]
        + df["sync_pulse_last_time_on.nanosec"] / 1e9
    )
    df["sync_pulse_last_time_off"] = (
        df["sync_pulse_last_time_off.sec"]
        + df["sync_pulse_last_time_off.nanosec"] / 1e9
    )
    df["safety_laser_last_time_broken"] = (
        df["safety_laser_last_time_broken.sec"]
        + df["safety_laser_last_time_broken.nanosec"] / 1e9
    )
    df["button_last_time_pressed"] = (
        df["button_last_time_pressed.sec"]
        + df["button_last_time_pressed.nanosec"] / 1e9
    )

    if verify:
        if freq_rtol is None or freq_var_tol is None:
            raise ValueError(
                "freq_rtol and freq_var_tol must be provided if verify is True"
            )

        # Convert timestamps to seconds
        df["bag_time"] = df.bag_time_ns / 1e9

        # Verify the timestamps
        verify_timestamps(
            df[["time", "bag_time"]],  # type: ignore
            freq,
            freq_rtol=freq_rtol,
            freq_var_tol=freq_var_tol,
        )

    df = cast(
        pd.DataFrame,
        df[["time", *TEENSY_DATA_COLS]],
    )

    return df


teensy_df = format_teensy_columns(raw_teensy_df, freq=100, verify=False)
teensy_df

In [ ]:
min_time = min(
    teensy_df[
        ["time", "sync_pulse_last_time_on", "sync_pulse_last_time_off"]
    ].min(axis=None),
    markers_df["time"].min(),
)
teensy_df[["time", "sync_pulse_last_time_on", "sync_pulse_last_time_off"]] = (
    teensy_df[["time", "sync_pulse_last_time_on", "sync_pulse_last_time_off"]]
    - min_time
)
markers_df["time"] = markers_df["time"] - min_time

In [ ]:
sync_on_idx = teensy_df["sync_pulse_last_time_on"].diff() > 1e-3
sync_on_times = teensy_df["sync_pulse_last_time_on"][sync_on_idx]
sync_off_idx = teensy_df["sync_pulse_last_time_off"].diff() > 1e-3
sync_off_times = teensy_df["sync_pulse_last_time_off"][sync_off_idx]
print(
    f"Sync pulse count | ON: {sync_on_times.count()}, OFF: {sync_off_times.count()}"
)
if SYNC_TRIGGER == "on":
    sync_times = sync_on_times.to_numpy()
elif SYNC_TRIGGER == "off":
    sync_times = sync_off_times.to_numpy()
else:
    sync_times = np.sort(np.concat([sync_on_times, sync_off_times]))

In [ ]:
if DIM == "all":
    cols = ["marker_x", "marker_y", "marker_z"]
else:
    cols = [f"marker_{DIM}"]

if USE_GRADIENT:
    markers_df["speed"] = np.linalg.norm(
        np.gradient(
            markers_df[cols],
            markers_df["time"],
            axis=0,
        ),
        axis=1,
    )
else:
    markers_df["speed"] = np.linalg.norm(
        (markers_df[cols].diff(axis=0)), axis=1
    )
markers_df["speed"] = markers_df[[f"marker_{DIM}"]].diff().abs()
markers_df["speed_zscore"] = zscore(markers_df["speed"], nan_policy="omit")
marker_moving = markers_df[markers_df["speed_zscore"] > 0.1]
time_diff = marker_moving["time"].diff()
marker_onsets = marker_moving[time_diff.isna() | (time_diff > 0.03)]
marker_times = np.sort(marker_onsets["time"].to_numpy())
marker_times.shape

In [ ]:
# min_time = min(sync_times.min(), marker_times.min())
# sync_times = sync_times - min_time
# marker_times = marker_times - min_time

In [ ]:
sync_times

In [ ]:
marker_times

In [ ]:
# diffs_easy = pd.Series(marker_times - sync_times[8:])
# diffs_easy.describe()

In [ ]:
sync_idx = len(sync_times) - 1
MARKER_IDX = len(marker_times) - 1
last_diff = None
last_times = None
times = []
diffs = []

while sync_idx >= 0 and MARKER_IDX >= 0:
    sync_time = sync_times[sync_idx]
    marker_time = marker_times[MARKER_IDX]

    diff = marker_time - sync_time
    if diff < 0:
        if last_diff is not None:
            assert last_times is not None
            diffs.append(last_diff)
            times.append(last_times)
            last_diff = None
            last_times = None
        sync_idx -= 1
    else:
        if last_diff is None or diff < last_diff:
            last_diff = diff
            last_times = (sync_time, marker_time)
        MARKER_IDX -= 1

# if last_diff is not None:
#     diffs.append(last_diff)
#     times.append(last_times)

diffs = pd.Series(list(reversed(diffs)))
times = list(zip(reversed(times)))
times = np.array(times).squeeze()
# assert np.isclose(diffs, diffs_easy).all()
assert (diffs > 0).all()
assert np.isclose(diffs, times[:, 1] - times[:, 0]).all()
diffs.describe()

In [ ]:
diffs

In [ ]:
# markers_y = (markers_df["marker_y"] - markers_df["marker_y"].min()) / (
#     markers_df["marker_y"].max() - markers_df["marker_y"].min()
# )
window = 0.3
start = 22.9
plt.plot(markers_df["time"], markers_df[f"marker_{DIM}"])
y = markers_df[markers_df["time"].isin(set(marker_times))][f"marker_{DIM}"]
plt.scatter(marker_times, y)
plt.scatter(sync_times, np.ones_like(sync_times) * -0.029)
plt.xlim(start, start + window)
# plt.plot(teensy_df["time"], teensy_df["sync_pulse_state"])